<a href="https://colab.research.google.com/github/loan1/btxrd-stage1-semisup/blob/main/notebooks/Stage1_BTXRD_SemiSup_SAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage-1 (BTXRD) — Semi-supervised + SAM/MedSAM (Generalist) + U-Net (Specialist)

Mục tiêu: khảo sát hành xử hệ thống khi mask ít, tận dụng unlabeled và kế thừa foundation model.

- Generalist (frozen): SAM/MedSAM tạo pseudo-mask cho tumor unlabeled
- Specialist (pixel-wise): U-Net train để deploy
- So sánh theo budget p% và full-label (100%) upper bound

Code nằm trong GitHub repo, dữ liệu/checkpoints nằm trong Google Drive.


In [ ]:
# 0) Mount Drive + Clone/Pull repo
from google.colab import drive
drive.mount("/content/drive")

import os
REPO_URL = "https://github.com/loan1/btxrd-stage1-semisup.git"
REPO_DIR = "/content/btxrd-stage1-semisup"

if not os.path.exists(REPO_DIR):
    print("Cloning repo...")
    !git clone {REPO_URL}
else:
    print("Repo exists, pulling latest...")
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
!ls


In [ ]:
# 1) Imports from repo
import sys
sys.path.append("/content/btxrd-stage1-semisup/src")

from config import RAW_DIR, PROC_DIR, PSEUDO_DIR, RUNS_DIR
from unet import UNet
from btxrd_dataset import BTXRDSegDataset
from train_utils import fit_posw_resume, eval_all_and_tumor_only, fp_on_normals, quick_stats

import torch, numpy as np, os, cv2, random
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device.startswith("cuda"):
    print("GPU:", torch.cuda.get_device_name(0))
torch.backends.cudnn.benchmark = True


In [ ]:
# 2) Seed
SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device.startswith("cuda"):
    torch.cuda.manual_seed_all(SEED)


## 3) Build train/val/test pools
Notebook đọc `processed/splits/val.txt` và `processed/splits/test.txt` (đã có).
Train pool = tất cả ảnh trừ val/test.
Tumor/normal phân biệt bằng mask GT (mask rỗng = normal).

In [ ]:
import glob

def read_list(p):
    return [x.strip() for x in open(p) if x.strip()]

val_ids  = read_list(f"{PROC_DIR}/splits/val.txt")
test_ids = read_list(f"{PROC_DIR}/splits/test.txt")

img_files = glob.glob(f"{RAW_DIR}/images/*")
all_ids = [os.path.basename(p) for p in img_files]

train_pool = [i for i in all_ids if i not in set(val_ids) and i not in set(test_ids)]

GT_MASK_DIR = f"{PROC_DIR}/masks_gt"
def stem(x): return os.path.splitext(x)[0]

def is_tumor_gt(img_id):
    m = cv2.imread(os.path.join(GT_MASK_DIR, stem(img_id)+".png"), 0)
    if m is None:
        raise FileNotFoundError(os.path.join(GT_MASK_DIR, stem(img_id)+".png"))
    return (m > 0).any()

train_tumor  = [i for i in train_pool if is_tumor_gt(i)]
train_normal = [i for i in train_pool if not is_tumor_gt(i)]

print("val:", len(val_ids), "test:", len(test_ids))
print("train tumor:", len(train_tumor), "train normal:", len(train_normal))


## 4) Budget split p%
p tính trên tumor train. Labeled dùng GT; unlabeled tumor dùng pseudo; normal luôn mask rỗng.

In [ ]:
def make_budget_split(train_tumor_ids, p, seed=2026):
    rng = np.random.default_rng(seed + int(p*1000))
    n = len(train_tumor_ids)
    k = max(1, int(round(n * p)))
    idx = rng.choice(n, size=k, replace=False)
    labeled = [train_tumor_ids[i] for i in idx]
    labeled_set = set(labeled)
    unlabeled = [x for x in train_tumor_ids if x not in labeled_set]
    return labeled, unlabeled


## 5) DataLoaders (supervised vs semi)
`normal_mult` tăng trọng số normal để giảm FP.
Pseudo dir mặc định: `pseudo/sam_box_oracle`.

In [ ]:
def build_loaders(p, pseudo_dir, img_size=512, batch=16, nw=2, normal_mult=3.0, mode="supervised"):
    labeled_tumor, unlabeled_tumor = make_budget_split(train_tumor, p=p, seed=SEED)

    ds_labeled = BTXRDSegDataset(f"{RAW_DIR}/images", labeled_tumor, "gt", GT_MASK_DIR, img_size=img_size)
    ds_normal  = BTXRDSegDataset(f"{RAW_DIR}/images", train_normal, "gt", GT_MASK_DIR, img_size=img_size)
    datasets = [ds_labeled, ds_normal]

    if mode == "semi":
        ds_unlab = BTXRDSegDataset(f"{RAW_DIR}/images", unlabeled_tumor, "pseudo", GT_MASK_DIR,
                                   pseudo_masks_dir=pseudo_dir, img_size=img_size)
        datasets = [ds_labeled, ds_unlab, ds_normal]

    train_ds = ConcatDataset(datasets)
    sizes = [len(d) for d in datasets]

    weights = []
    weights += [1.0] * sizes[0]
    if mode == "semi":
        weights += [1.0] * sizes[1]
        weights += [normal_mult] * sizes[2]
    else:
        weights += [normal_mult] * sizes[1]

    sampler = WeightedRandomSampler(weights, num_samples=sum(sizes), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=batch, sampler=sampler,
                              num_workers=nw, pin_memory=True, persistent_workers=(nw>0))

    val_ds  = BTXRDSegDataset(f"{RAW_DIR}/images", val_ids,  "gt", GT_MASK_DIR, img_size=img_size)
    test_ds = BTXRDSegDataset(f"{RAW_DIR}/images", test_ids, "gt", GT_MASK_DIR, img_size=img_size)

    val_loader  = DataLoader(val_ds,  batch_size=batch, shuffle=False, num_workers=nw, pin_memory=True, persistent_workers=(nw>0))
    test_loader = DataLoader(test_ds, batch_size=batch, shuffle=False, num_workers=nw, pin_memory=True, persistent_workers=(nw>0))

    return train_loader, val_loader, test_loader, {"labeled_tumor": len(labeled_tumor), "unlabeled_tumor": len(unlabeled_tumor), "normal": len(train_normal)}


## 6) Train/Resume + Evaluate
Lưu checkpoints vào Drive: `RUNS_DIR/<exp>/best.pth` và `last.pt` (resume sau crash).

In [ ]:
def run_experiment(exp_name, mode, p, pseudo_subdir="sam_box_oracle",
                   pos_weight=15.0, normal_mult=3.0,
                   img_size=512, batch=16, nw=2,
                   lr=1e-3, epochs_total=20, thr_val=0.5, resume=True):

    pseudo_dir = f"{PSEUDO_DIR}/{pseudo_subdir}"
    out_dir = f"{RUNS_DIR}/{exp_name}"
    os.makedirs(out_dir, exist_ok=True)

    train_loader, val_loader, test_loader, info = build_loaders(
        p=p, pseudo_dir=pseudo_dir, img_size=img_size, batch=batch, nw=nw,
        normal_mult=normal_mult, mode=mode
    )
    print("Split info:", info)
    print("Out:", out_dir)

    model = UNet(1,1,32).to(device)
    fit_posw_resume(model, train_loader, val_loader, device, out_dir,
                    pos_weight=pos_weight, lr=lr, epochs_total=epochs_total, thr_val=thr_val, resume=resume)

    best_path = f"{out_dir}/best.pth"
    best = UNet(1,1,32).to(device)
    best.load_state_dict(torch.load(best_path, map_location=device))

    val_met  = eval_all_and_tumor_only(best, val_loader, device, thr=thr_val)
    test_met = eval_all_and_tumor_only(best, test_loader, device, thr=thr_val)
    val_fp   = fp_on_normals(best, val_loader, device, thr=thr_val)
    test_fp  = fp_on_normals(best, test_loader, device, thr=thr_val)

    print("VAL:", val_met, "VAL FP:", val_fp)
    print("TEST:", test_met, "TEST FP:", test_fp)

    return {"exp": exp_name, "mode": mode, "p": p, "pos_weight": pos_weight, "normal_mult": normal_mult, "thr": thr_val,
            **{f"val_{k}": v for k,v in val_met.items()}, **{f"test_{k}": v for k,v in test_met.items()},
            **{f"val_{k}": v for k,v in val_fp.items()}, **{f"test_{k}": v for k,v in test_fp.items()}}


## 7) Run p=10 baseline (supervised vs semi)
Chạy 2 thí nghiệm này trước để kiểm tra pipeline. Có thể thay `pos_weight` và `normal_mult`.

In [ ]:
import pandas as pd

results = []
results.append(run_experiment("p10_supervised_posw15_norm3", "supervised", p=0.10, pos_weight=15.0, normal_mult=3.0, epochs_total=20, thr_val=0.5, resume=True))
results.append(run_experiment("p10_semi_sam_posw15_norm3", "semi", p=0.10, pos_weight=15.0, normal_mult=3.0, epochs_total=20, thr_val=0.5, resume=True))

df = pd.DataFrame(results)
df


## 8) Save summary to Drive (CSV)

In [ ]:
import pandas as pd, os

summary_path = os.path.join(RUNS_DIR, "summary_stage1.csv")
df = pd.DataFrame(results)

if os.path.exists(summary_path):
    old = pd.read_csv(summary_path)
    df_out = pd.concat([old, df], ignore_index=True)
else:
    df_out = df

df_out.to_csv(summary_path, index=False)
print("Saved:", summary_path)
df_out.tail(10)


## 9) Sweep p = 1/5/10/20 + full-label upper bound
Sau khi p=10 ổn, bật khối dưới để chạy sweep. Full-label: p=1.0 (tumor labeled = all tumor).

In [ ]:
# Uncomment to run sweeps
# import pandas as pd, os
# p_list = [0.01, 0.05, 0.10, 0.20, 1.0]
# modes = ["supervised", "semi"]
# results = []
# for p in p_list:
#     for mode in modes:
#         exp = f"p{int(p*100):02d}_{mode}_posw15_norm3" if mode=="supervised" else f"p{int(p*100):02d}_{mode}_sam_posw15_norm3"
#         results.append(run_experiment(exp, mode, p=p, pos_weight=15.0, normal_mult=3.0, epochs_total=20, thr_val=0.5, resume=True))
#
# df = pd.DataFrame(results)
# out_csv = os.path.join(RUNS_DIR, "summary_sweep.csv")
# df.to_csv(out_csv, index=False)
# print("Saved:", out_csv)
# df
